![Module 3: Self-Modifying Prompt](../images/module-3-self-modify.svg)

# Module 3: Self-Modifying Prompt

> The agent rewrites its own system prompt. Teach it once ("respond in haiku"); it persists to disk and survives restarts. **Behavior change, not just context.**

📖 Full explainer: see the companion page in `content/` of this repo.  
💻 Standalone script: `code/step-0?/agent.py`  
🔗 Workshop: https://github.com/strands-agents/samples/tree/main/python/01-learn/18-self-improving-agents

---


# AIM308 — Module 2: Self-Modifying System Prompt

In [ ]:
%pip install -q strands-agents strands-agents-tools

In [ ]:
import os

MODEL_ID = "global.anthropic.claude-opus-4-8"  # Claude Sonnet 4.5
os.environ["AWS_REGION"] = "us-east-1"


## The `system_prompt` tool — agent writes to its own env var and a `.prompt` file

In [ ]:
from pathlib import Pathfrom strands import toolPROMPT_FILE = Path(".prompt")@tooldef system_prompt(action: str, prompt: str | None = None) -> dict:    """Manage the agent's own system prompt."""    if action == "view":        return {"status": "success", "content": [{"text": os.environ.get("SYSTEM_PROMPT","(empty)")}]}    if action == "update":        os.environ["SYSTEM_PROMPT"] = prompt        PROMPT_FILE.write_text(prompt)        return {"status": "success", "content": [{"text": "Prompt updated & persisted."}]}    if action == "reset":        os.environ.pop("SYSTEM_PROMPT", None)        if PROMPT_FILE.exists(): PROMPT_FILE.unlink()        return {"status": "success", "content": [{"text": "Reset."}]}    return {"status": "error", "content": [{"text": "Unknown action"}]}

## Dynamic system-prompt construction

In [ ]:
from datetime import datetimedef build_system_prompt():    base = "You are a self-improving research agent. You can modify your own system prompt."    persistent = PROMPT_FILE.read_text() if PROMPT_FILE.exists() else ""    env_ext = os.environ.get("SYSTEM_PROMPT", "")    runtime = f"Time: {datetime.now().isoformat(timespec='seconds')}"    return "\n".join(filter(None, [base, persistent, env_ext, runtime]))

## Build the agent

In [ ]:
from strands import Agentfrom strands_tools import shelldef new_agent():    return Agent(        model=MODEL_ID,        tools=[shell, system_prompt],        system_prompt=build_system_prompt(),    )

## Teach it to respond in haiku

In [ ]:
new_agent()("view your system prompt")

In [ ]:
new_agent()("from now on always reply in haiku — update your prompt permanently")

In [ ]:
new_agent()("what's 2+2?")  # should be a haiku

## Reset

In [ ]:
new_agent()("reset your system prompt")

Next: **[03_memory.ipynb](03_memory.ipynb)**